# 13 · One development set, one ruler, every model and context

**The decision:** which combination passes the predeclared quality bar, and what does each technique add?

Five saved models share six conditions: **naive**, **engineered**, **all documents**, **similarity retrieval**, **reranked retrieval**, and **oracle evidence**. Each normal condition uses all 50 `dev_v1` questions; oracle uses its 30 answerable questions. That is **1,400 answers** when all five checkpoints exist. Optional `trained_prompt` adds a seventh condition using the short prompt from SFT training.

Change **`configs/master_eval.yaml`**, then run top to bottom in a fresh **Python 3 (Post-Training-Lab)** kernel. Default `run: false` checks files and previews the plan without loading neural-network weights or querying PostgreSQL. No external APIs, training, adapter merging, or final-set evaluation occurs here. This notebook only reads existing checkpoints.

The user question is identical across conditions; only the system message/evidence block changes. Models are freshly loaded one at a time. Retrieval is prepared for the whole selected evaluation set once, saved, and released before answering begins.

In [ ]:
# Standard libraries handle paths, JSON, hashes, timing, cleanup and local service commands.
# YAML reads settings; pandas/display make tables. Importing lab/rag does not load neural networks.
from pathlib import Path
import sys, json, hashlib, time, gc, warnings, copy, ast, math, subprocess, shutil
from datetime import datetime, timezone
from collections import Counter
from itertools import combinations
from statistics import NormalDist
import importlib.metadata as metadata
import yaml
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/lab.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src import lab, rag
CFG = lab.cfg()
CONFIG_PATH = ROOT / "configs/master_eval.yaml"
MASTER = yaml.safe_load(CONFIG_PATH.read_text())

In [ ]:
# These helpers normalize paths and hash exact content. A hash changes when its input changes.
# A smoke harness can supply MASTER_OVERRIDES in memory; normal use edits only the YAML file.
def project_path(value):
    path = Path(value).expanduser()
    return (path if path.is_absolute() else ROOT / path).resolve()

def digest(value):
    text = json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False)
    return hashlib.sha256(text.encode()).hexdigest()

def file_sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def deep_update(target, changes):
    for key, value in changes.items():
        if isinstance(value, dict) and isinstance(target.get(key), dict):
            deep_update(target[key], value)
        else:
            target[key] = copy.deepcopy(value)
    return target
MASTER = deep_update(MASTER, globals().get("MASTER_OVERRIDES", {}))

In [ ]:
# Validate the selected names before doing expensive work. Optional conditions remain off unless selected.
RUN, SELECTED, CONDITIONS = MASTER["run"], MASTER["selected_models"], MASTER["conditions"]
SPECS, GEN = MASTER["models"], MASTER["generation"]
assert isinstance(RUN, bool)
assert SELECTED and len(SELECTED) == len(set(SELECTED)) and set(SELECTED) <= set(SPECS)
assert CONDITIONS and len(CONDITIONS) == len(set(CONDITIONS))
assert set(CONDITIONS) <= {"naive", "engineered", "with_docs", "rag_similarity", "rag_rerank", "oracle"} | set(MASTER["optional_conditions"])
assert set(MASTER["optional_conditions"]) <= {"trained_prompt"}
assert GEN["do_sample"] is False and GEN["num_beams"] == 1
GEN["device"] = lab.resolve_device(GEN["device"])   # auto -> cuda, mps or cpu
assert GEN["dtype"] == "float32", "Every model is evaluated in float32"
assert GEN["max_new_tokens"] > 0 and GEN["max_input_tokens"] > 0
assert 0 < MASTER["reporting"]["confidence"] < 1
assert MASTER["cache_tag"] and isinstance(MASTER["cache_tag"], str)
OUTPUT_ROOT = project_path(MASTER["output_directory"])
CACHE_DIR = project_path(MASTER["cache_directory"])
VERSIONS = {name: metadata.version(name) for name in ["torch", "transformers", "peft", "psycopg"]}
print("Run:", RUN, "| models:", SELECTED, "| conditions:", CONDITIONS)
print("Cache tag:", MASTER["cache_tag"], "| outputs:", OUTPUT_ROOT)

## 1. Inspect the new development set

`dev_v1` targets factual reading and refusal generalization: 30 single-fact answerable questions and 20 unanswerable questions, including price/spec/policy near misses. The manifest records authoring and overlap checks. It is informed by earlier dev failures, so it is still a **development set**.

The answerable questions exclude colour lists and Arc's numeric-zero fast-charge label because the unchanged scorer mishandles those cases. This improves the measurement design; it does not repair the scorer. Selecting a smaller cohort is useful for plumbing checks, not for ranking models.

In [ ]:
# Resolve the evaluation name through the shared registry. Read only that registered development file.
EVAL_NAME = MASTER["eval_set"]
registry = CFG["eval_sets"]["sets"]
assert EVAL_NAME in registry, "Register this development set in configs/config.yaml first"
assert EVAL_NAME != "heldout", "The final set stays reserved for notebook 08"
EVAL_PATH = project_path(registry[EVAL_NAME]["path"])
assert EVAL_PATH.name != "heldout.json"
all_items = json.loads(EVAL_PATH.read_text())
assert len(all_items) == registry[EVAL_NAME]["n"]
assert len({q["id"] for q in all_items}) == len(all_items)
fidx = lab.fact_index()
for q in all_items:
    assert isinstance(q["answerable"], bool) and q["question"].strip()
    if q["answerable"]:
        fact = fidx[q["fact_id"]]
        assert q["expected_value"] == fact["value"] and q["unit"] == fact.get("unit")
    else:
        assert q["fact_id"] is None and q["expected_value"] is None
by_id = {q["id"]: q for q in all_items}
items = all_items
if MASTER["question_ids"] is not None:
    ids = MASTER["question_ids"]
    assert ids and len(ids) == len(set(ids)) and set(ids) <= set(by_id)
    items = [by_id[qid] for qid in ids]
limit = MASTER["max_questions"]
assert limit is None or (type(limit) is int and 0 < limit <= len(items))
items = items if limit is None else items[:limit]
RESULT_DIR = OUTPUT_ROOT / EVAL_NAME
print(len(all_items), "registered questions;", len(items), "selected for generation")

In [ ]:
# Show the full-set balance, not just a smoke prefix. Extra schema fields stay available for diagnostics.
question_table = pd.DataFrame(all_items)
display(question_table.groupby(["answerable", "category"]).size().rename("questions").reset_index())
display(question_table.groupby(["answerable", "style"]).size().rename("questions").reset_index())
for category, group in question_table.groupby("category", sort=False):
    print("\nCategory:", category)
    display(group[["id", "question", "answerable", "style"]].head(MASTER["reporting"]["examples_per_category"]))
MANIFEST_PATH = project_path(MASTER["manifest_path"])
manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else None
if manifest and EVAL_NAME == manifest.get("name"):
    assert manifest["dataset_sha256"] == file_sha(EVAL_PATH), "Dataset/manifest mismatch: rebuild dev_v1"
print("Authoring manifest:")
print(json.dumps(manifest, indent=2, ensure_ascii=False) if manifest else "No manifest configured/found for this set")

## 2. Fix the prompt bytes and the measurement identity

We read ENGINEERED from the saved notebook-12 protocol, not notebook 05's slightly different evaluation wording. The optional trained prompt is the literal `SYSTEM` assignment from the SFT notebook. Parsing that assignment reads text; it does not execute the training notebook.

Caches include weight contents, exact system/user text, generation settings, tokenizer/config files, question labels/schema, scorer/code versions, and retrieval provenance. Changing a comment or saved notebook output does not force new answers. Failed, empty, or truncated attempts are recorded but never reused as successful cached answers.

In [ ]:
# Read exact prompt strings from configs/master_eval.yaml without executing another notebook.
ENGINEERED = MASTER["prompt_sources"]["engineered"]
NAIVE = MASTER["prompt_sources"]["naive"]
SEPARATOR = MASTER["prompt_sources"]["separator"]
assert SEPARATOR == "\n\n=== MERIDIAN OFFICIAL DOCUMENTS ===\n"
DOCUMENT_PATHS = sorted(lab.RAW.glob("*.md"))
DOCUMENTS = [p.read_text() for p in DOCUMENT_PATHS]
DOCS = "\n\n".join(DOCUMENTS)
assert DOCUMENTS and ENGINEERED.strip()
TRAINED_PROMPT = None
if "trained_prompt" in CONDITIONS:
    trained_source = json.loads(project_path(MASTER["prompt_sources"]["trained_notebook"]).read_text())
    matches = []
    for cell in trained_source["cells"]:
        if cell["cell_type"] == "code":
            for node in ast.parse("".join(cell["source"])).body:
                if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == MASTER["prompt_sources"]["trained_variable"] for t in node.targets):
                    matches.append(ast.literal_eval(node.value))
    assert len(matches) == 1 and isinstance(matches[0], str)
    TRAINED_PROMPT = matches[0]
print("ENGINEERED sha256:", hashlib.sha256(ENGINEERED.encode()).hexdigest(), "| source: configs/master_eval.yaml")

In [ ]:
# Combine executable statements before hashing: comments, cell splits and output metadata do not matter.
notebook = json.loads((ROOT / "notebooks/13_master_eval.ipynb").read_text())
nodes = []
for cell in notebook["cells"]:
    if cell["cell_type"] == "code":
        nodes.extend(ast.parse("".join(cell["source"])).body)
CODE_HASH = digest(ast.dump(ast.Module(body=nodes, type_ignores=[]), include_attributes=False))
PROVENANCE = {
    "schema_version": "master-eval-v1", "eval_set": EVAL_NAME,
    "eval_file_sha256": file_sha(EVAL_PATH), "facts_sha256": file_sha(lab.RAW / "facts.json"),
    "scorer_sha256": file_sha(ROOT / "src/lab.py"), "notebook_code_sha256": CODE_HASH,
    "versions": VERSIONS, "documents": {p.name: file_sha(p) for p in DOCUMENT_PATHS},
    "engineered_sha256": hashlib.sha256(ENGINEERED.encode()).hexdigest(), "master_yaml_sha256": file_sha(CONFIG_PATH),
}
# Execution controls/output paths are recorded, but are not themselves generation inputs.
CACHE_PROVENANCE = {key: value for key, value in PROVENANCE.items() if key != "master_yaml_sha256"}

## 3. Check which saved models exist

These checks read configurations and hash files; they do not instantiate an LLM. A missing or incompatible artifact becomes **SKIPPED** with its path. Each adapter must name the exact configured base. A merged checkpoint is loaded as a full model, with no additional adapter.

For **SFT + DPO**, load only the root `adapters/dpo` policy. `adapters/dpo/ref/` is TRL's frozen copy of SFT and is not an additional inference adapter. The runtime also asserts that the only attached adapter is `default`.

In [ ]:
# A full model can store weights in one file or in shards named by an index file.
def checkpoint_files(path):
    if not (path / "config.json").is_file() or (path / "adapter_config.json").exists():
        raise ValueError(f"Not a complete model checkpoint: {path}")
    for name in ["model.safetensors", "pytorch_model.bin"]:
        if (path / name).is_file():
            return [path / name]
    for name in ["model.safetensors.index.json", "pytorch_model.bin.index.json"]:
        index = path / name
        if index.is_file():
            shards = [path / name for name in sorted(set(json.loads(index.read_text())["weight_map"].values()))]
            if shards and all(p.is_file() for p in shards):
                return [index, *shards]
    raise ValueError(f"Missing complete model weights: {path}")

In [ ]:
# Hash every checkpoint weight file plus tokenizer/config files. Adapters include their base identity.
# The in-memory file-hash memo avoids rereading the same unchanged base for several adapters.
FILE_HASHES = {}
def artifact_hash(path):
    state = (str(path.resolve()), path.stat().st_size, path.stat().st_mtime_ns)
    if state not in FILE_HASHES:
        FILE_HASHES[state] = file_sha(path)
    return FILE_HASHES[state]

In [ ]:
# Inspect one declared checkpoint and ensure each adapter belongs to its selected base.
def inspect_model(model_id):
    spec = copy.deepcopy(SPECS[model_id])
    assert spec["kind"] in {"local_base", "local_adapter", "local_merged"}
    base = project_path(spec.get("base_path") or CFG["model"]["base_path"])
    tokenizer = project_path(spec.get("tokenizer_path") or str(base))
    weights = checkpoint_files(base)
    if not (tokenizer / "tokenizer_config.json").is_file():
        raise ValueError(f"Missing tokenizer: {tokenizer}")
    metadata_files = sorted({*base.glob("*.json"), *tokenizer.glob("*.json"), *tokenizer.glob("*.txt"),
                             *tokenizer.glob("*.jinja"), *tokenizer.glob("*.model"), *tokenizer.glob("*.tiktoken")})
    identity = {"base_path": str(base), "base_weights": {str(p): artifact_hash(p) for p in weights},
                "metadata": {str(p): artifact_hash(p) for p in metadata_files}, "adapter": None}
    spec.update(base_path=str(base), tokenizer_path=str(tokenizer))
    if spec["kind"] == "local_adapter":
        adapter = project_path(spec["adapter_path"])
        config_file = adapter / "adapter_config.json"
        weight = next((adapter / n for n in ["adapter_model.safetensors", "adapter_model.bin"] if (adapter / n).is_file()), None)
        if not config_file.is_file() or weight is None:
            raise ValueError(f"Missing adapter config/weights: {adapter}")
        settings = json.loads(config_file.read_text())
        if settings.get("peft_type") != "LORA" or settings.get("task_type") != "CAUSAL_LM":
            raise ValueError(f"Expected causal-LM LoRA adapter: {adapter}")
        if project_path(settings["base_model_name_or_path"]) != base:
            raise ValueError(f"Adapter base does not match selected base: {adapter}")
        if model_id == "sft_dpo":
            assert adapter == (lab.ADAPTERS / "dpo").resolve(), "Load the DPO root, not ref/"
        spec["adapter_path"] = str(adapter)
        identity["adapter"] = {"path": str(adapter), "config_sha256": artifact_hash(config_file),
                               "weight_files": {str(weight): artifact_hash(weight)},
                               "recorded_base": settings["base_model_name_or_path"]}
    identity["compound_sha256"] = digest(identity)
    return spec, identity

In [ ]:
# Keep missing candidates visible rather than silently substituting the base model.
READY, INVENTORY = {}, []
for model_id in SELECTED:
    try:
        spec, identity = inspect_model(model_id)
        READY[model_id] = {"spec": spec, "artifact": identity}
        state, detail = "READY", spec.get("adapter_path") or spec["base_path"]
    except (OSError, ValueError, KeyError, AssertionError) as exc:
        state, detail = "SKIPPED", str(exc)
    INVENTORY.append({"model": model_id, "label": SPECS[model_id]["label"], "status": state, "path_or_reason": detail})
display(pd.DataFrame(INVENTORY))

## 4. Prepare one evidence snapshot for all models

The local BGE bi-encoder embeds each question, PostgreSQL returns vector candidates, and the cross-encoder reranks those same candidates. We retain both orders. Gold fact IDs measure retrieval afterward; they never enter search or reranking.

This notebook's named similarity conditions use vector-only candidates and an exact top-K selection. A rejection threshold would change that condition, so it must be null here. The snapshot includes actual index content and source hashes. If the stored index differs from current source chunks, rebuild it in notebook 11.

In [ ]:
# Use a private copy of retrieval settings and a notebook-14 cache directory.
RAG = copy.deepcopy(rag.load_config(project_path(MASTER["rag_config_path"])))
assert RAG["index_name"] == MASTER["retrieval"]["required_index"]
assert RAG["rerank"]["enabled"], "The master comparison requires reranking"
assert RAG["retrieval"]["min_rerank_score"] is None, "Exact top-K conditions require no rejection threshold"
if MASTER["retrieval"]["force_vector_only"]:
    RAG["retrieval"]["hybrid"]["enabled"] = False
assert not RAG["retrieval"]["hybrid"]["enabled"], "Similarity conditions must use vector-only candidates"
RAG["cache"]["directory"] = str(project_path(MASTER["retrieval_cache_directory"]))
assert MASTER["retrieval"]["require_exact_top_k"] is True
TOP_K, TOP_N = RAG["retrieval"]["top_k"], RAG["retrieval"]["top_n_candidates"]
RAG_SETTINGS = {key: RAG[key] for key in ["index_name", "corpus", "chunking", "embedding", "rerank", "retrieval"]}
NEEDS_RETRIEVAL = bool(set(CONDITIONS) & {"rag_similarity", "rag_rerank", "oracle"})
CHUNKS, RETRIEVAL, INDEX_HASH, RETRIEVAL_HASH = [], {}, None, None
RETRIEVAL_STATS, RETRIEVAL_PATH = {}, None

In [ ]:
# JSON writes are atomic. Existing result files with another cache tag are never overwritten.
def now():
    return datetime.now(timezone.utc).isoformat()

def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + "\n")
    temporary.replace(path)

def guard_tag(path):
    if path.exists() and json.loads(path.read_text()).get("cache_tag") != MASTER["cache_tag"]:
        raise ValueError(f"Different cache_tag already owns {path}. Choose a new output_directory; old results are preserved.")

def save_result(path, payload):
    guard_tag(path)
    if path.exists():
        old = json.loads(path.read_text())
        if old.get("identity") != payload.get("identity"):
            archive = path.parent / "history" / (path.stem + "__" + digest(old)[:16] + ".json")
            archive.parent.mkdir(parents=True, exist_ok=True)
            if not archive.exists():
                shutil.copy2(path, archive)
    write_json(path, {**payload, "cache_tag": MASTER["cache_tag"], "saved_at": now()})

In [ ]:
# Start only the configured local PostgreSQL service if needed; never change databases or roles.
# This branch stays inactive in preview. The index itself must already have been built in notebook 11.
if RUN and NEEDS_RETRIEVAL and RAG["database"]["dsn"] == MASTER["postgres"]["local_dsn"]:
    ready = subprocess.run(MASTER["postgres"]["readiness_command"], capture_output=True, text=True)
    if ready.returncode != 0:
        if not MASTER["postgres"]["start_if_needed"]:
            raise RuntimeError("Start PostgreSQL with the configured local service command before continuing")
        started = subprocess.run(MASTER["postgres"]["start_command"], capture_output=True, text=True)
        if started.returncode != 0:
            raise RuntimeError("The configured PostgreSQL service command failed; inspect the service locally")
        deadline = time.monotonic() + MASTER["postgres"]["startup_timeout_seconds"]
        while subprocess.run(MASTER["postgres"]["readiness_command"], capture_output=True).returncode != 0:
            if time.monotonic() >= deadline:
                raise RuntimeError("PostgreSQL did not become ready within the configured startup timeout")
            time.sleep(0.5)

In [ ]:
# Compare indexed metadata with source chunks. This loads tokenizers, not answering models.
# Row order is irrelevant: dictionaries keyed by chunk_id form the canonical comparison.
CHUNK_FIELDS = ["chunk_id", "doc_id", "source_path", "heading_path", "chunk_text", "embed_text",
                "token_count", "fact_ids", "content_sha256", "source_refs"]
def canonical_chunk(chunk):
    return {key: chunk[key] for key in CHUNK_FIELDS}

def canonical(chunks):
    return {c["chunk_id"]: canonical_chunk(c) for c in chunks}
if RUN and NEEDS_RETRIEVAL:
    expected, indexed = canonical(rag.chunk_corpus(RAG)), canonical(rag.all_chunks(RAG))
    if expected != indexed:
        raise ValueError("Index/source mismatch: rebuild notebook 11 before master evaluation")
    CHUNKS = [indexed[key] for key in sorted(indexed)]
    INDEX_HASH = digest(indexed)
    for question in all_items:
        if question["answerable"]:
            assert any(question["fact_id"] in c["fact_ids"] for c in CHUNKS), question["id"]
    RETRIEVAL_HASH = digest({"index": INDEX_HASH, "settings": RAG_SETTINGS, "questions": all_items,
                             "rag_source_sha256": file_sha(ROOT / "src/rag.py")})
    print(len(CHUNKS), "chunks match source; index fingerprint:", INDEX_HASH[:16])
else:
    print("Index checks deferred; no retrieval/database activity in preview.")

In [ ]:
# Save evidence once for the ENTIRE registered set, even when generation uses a smoke subset.
# finally releases BGE before a later cell can load Qwen; cached evidence must match its full identity.
if RUN and NEEDS_RETRIEVAL:
    RETRIEVAL_PATH = RESULT_DIR / ("retrieval__" + RETRIEVAL_HASH + ".json")
    before_stats = rag.stats()
    try:
        if RETRIEVAL_PATH.exists():
            saved = json.loads(RETRIEVAL_PATH.read_text())
            assert saved["identity"] == RETRIEVAL_HASH
            RETRIEVAL = saved["questions"]
            assert set(RETRIEVAL) == set(by_id)
        else:
            for question in all_items:
                candidates = rag.search_candidates(question["question"], RAG)
                ranked = rag.rerank_candidates(question["question"], candidates, RAG)
                assert len(candidates) >= TOP_K and len(ranked) >= TOP_K
                gold = [c for c in CHUNKS if question["answerable"] and question["fact_id"] in c["fact_ids"]]
                RETRIEVAL[question["id"]] = {"question": question["question"], "candidates": candidates,
                    "reranked": ranked, "similarity": candidates[:TOP_K], "rerank": ranked[:TOP_K],
                    "gold_chunks": gold, "gold_chunk_ids": [c["chunk_id"] for c in gold]}
            write_json(RETRIEVAL_PATH, {"identity": RETRIEVAL_HASH, "index_hash": INDEX_HASH,
                       "retrieval_settings": RAG_SETTINGS, "questions": RETRIEVAL, "saved_at": now()})
    finally:
        rag.release_models()
        gc.collect()
    RETRIEVAL_STATS = {key: value - before_stats[key] for key, value in rag.stats().items()}
    print("Evidence ready for", len(RETRIEVAL), "questions; BGE weights released.")
else:
    print("Retrieval not executed.")

In [ ]:
# Recall@k asks whether any gold chunk occurs in the first k results; MRR rewards earlier gold ranks.
retrieval_rows, retrieval_misses = [], []
if RETRIEVAL:
    answerable = [q for q in all_items if q["answerable"]]
    for stage, field in [("similarity", "candidates"), ("rerank", "reranked")]:
        ranks = []
        for q in answerable:
            evidence = RETRIEVAL[q["id"]]
            gold = set(evidence["gold_chunk_ids"])
            rank = next((i for i, c in enumerate(evidence[field], 1) if c["chunk_id"] in gold), None)
            ranks.append(rank)
            if rank is None or rank > TOP_K:
                retrieval_misses.append({"stage": stage, "question_id": q["id"], "question": q["question"], "gold_rank": rank})
        count = len(ranks)
        retrieval_rows.append({"stage": stage, "n_answerable": count,
            "recall_at_1": sum(r is not None and r <= 1 for r in ranks)/count,
            "recall_at_3": sum(r is not None and r <= 3 for r in ranks)/count,
            "MRR": sum(1/r if r is not None else 0 for r in ranks)/count,
            "candidate_pool": TOP_N})
    display(pd.DataFrame(retrieval_rows))
    display(pd.DataFrame(retrieval_misses))
    print("MRR is measured within the retrieved candidate pool; a missing gold chunk contributes zero.")

## 5. Build the plan and check every prompt before loading an answer model

A request means one model answering one question under one condition. Oracle excludes unknown questions because there is no gold evidence to supply. A missing model gets no requests. The time estimate uses earlier local measurements and excludes retrieval, checkpoint loading and reporting.

For a short plumbing check, set `question_ids`, `selected_models`, and `conditions` in YAML. This reduces **answer generation**; retrieval still measures the complete registered development set. Use a separate output/cache directory for a smoke test.

In [ ]:
# Assemble evidence in its stored order. Headers make each chunk boundary inspectable.
def format_chunks(chunks):
    return "\n\n".join(f"[chunk {c['chunk_id']} | {c['doc_id']}]\n{c['embed_text']}" for c in chunks)

def prompt_for(q, condition):
    chunks = []
    if condition == "naive":
        system = NAIVE
    elif condition == "engineered":
        system = ENGINEERED
    elif condition == "trained_prompt":
        system = TRAINED_PROMPT
    elif condition == "with_docs":
        system = ENGINEERED + SEPARATOR + DOCS
        assert all(document in system for document in DOCUMENTS)
    else:
        evidence = RETRIEVAL[q["id"]]
        field = {"rag_similarity": "similarity", "rag_rerank": "rerank", "oracle": "gold_chunks"}[condition]
        chunks = evidence[field]
        assert chunks and len({c["chunk_id"] for c in chunks}) == len(chunks)
        if condition.startswith("rag_"):
            assert len(chunks) == TOP_K
        else:
            assert q["answerable"] and all(q["fact_id"] in c["fact_ids"] for c in chunks)
        system = ENGINEERED + SEPARATOR + format_chunks(chunks)
    if condition in {"naive", "engineered", "trained_prompt"}:
        assert SEPARATOR not in system
    return system, chunks

def cohort(condition):
    return [q for q in items if condition != "oracle" or q["answerable"]]

In [ ]:
# Count eligible requests before loading any answer model; cache hits may shorten this estimate.
PLAN = [{"model": model_id, "condition": condition, "requests": len(cohort(condition)),
         "estimated_seconds": len(cohort(condition))*MASTER["timing_estimates_seconds"][condition]}
        for model_id in SELECTED if model_id in READY for condition in CONDITIONS]
display(pd.DataFrame(PLAN))
print(sum(r["requests"] for r in PLAN), "planned answers | estimated generation minutes:",
      round(sum(r["estimated_seconds"] for r in PLAN)/60, 1))
print("Estimate excludes retrieval and checkpoint loading; successful cache hits avoid generation.")

In [ ]:
# A job's identity includes context and question labels as well as weights and exact prompt text.
# Condition names distinguish reporting rows even if two conditions happen to contain identical evidence.
JOBS, TOKENIZERS, OUTPUT_PATHS = {}, {}, {}
def prepare_job(model_id, condition, q):
    system, chunks = prompt_for(q, condition)
    identity = {"provenance": CACHE_PROVENANCE, "cache_tag": MASTER["cache_tag"],
        "model_weights": READY[model_id]["artifact"], "model_kind": READY[model_id]["spec"]["kind"],
        "condition": condition, "system_prompt": system, "question": q, "generation": GEN,
        "chunks": [canonical_chunk(c) for c in chunks],
        "retrieval_identity": RETRIEVAL_HASH if condition in {"rag_similarity", "rag_rerank", "oracle"} else None}
    return {"model": model_id, "condition": condition, "item": q, "system_prompt": system,
            "chunks": chunks, "identity": identity, "key": digest(identity)}

def reusable(saved, job):
    return (saved.get("identity") == job["identity"] and saved.get("key") == job["key"]
            and saved.get("status") == "ok" and bool(saved.get("answer", "").strip()))

def read_cached(job):
    path = CACHE_DIR / (job["key"] + ".json")
    if path.exists():
        try:
            saved = json.loads(path.read_text())
            if reusable(saved, job):
                return saved
        except (OSError, ValueError):
            pass
    return None

if RUN:
    # Refuse a conflicting cache tag before any model starts answering.
    for model_id in SELECTED:
        for condition in CONDITIONS:
            path = RESULT_DIR / f"{model_id}__{condition}.json"
            guard_tag(path)
            OUTPUT_PATHS[(model_id, condition)] = path
    guard_tag(RESULT_DIR / "summary.json")
    JOBS = {(model_id, condition): [prepare_job(model_id, condition, q) for q in cohort(condition)]
            for model_id in READY for condition in CONDITIONS}
else:
    print("Preview only: no retrieval, tokenizer/model loading, generation, or output writes.")

In [ ]:
# Tokenizers turn messages into IDs without loading model weights. Validate ALL selected prompts first.
# truncation=False preserves the full prompt; an oversized prompt stops the run with a clear error.
TOKEN_ROWS = []
if RUN:
    from transformers import AutoTokenizer
    for model_id in READY:
        tokenizer = AutoTokenizer.from_pretrained(READY[model_id]["spec"]["tokenizer_path"], local_files_only=True)
        TOKENIZERS[model_id] = tokenizer
        for condition in CONDITIONS:
            for job in JOBS[(model_id, condition)]:
                messages = [{"role": "system", "content": job["system_prompt"]},
                            {"role": "user", "content": job["item"]["question"]}]
                ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=False)
                assert isinstance(ids, list) and all(type(token) is int for token in ids), "Expected one flat list of token IDs"
                if len(ids) > GEN["max_input_tokens"]:
                    raise ValueError(f"Prompt too long: {model_id}/{condition}/{job['item']['id']}: {len(ids)} > {GEN['max_input_tokens']}; nothing was truncated")
                job["input_ids"], job["prompt_tokens"] = ids, len(ids)
                TOKEN_ROWS.append({"model": model_id, "condition": condition, "prompt_tokens": len(ids),
                                   "cached": read_cached(job) is not None})
    token_frame = pd.DataFrame(TOKEN_ROWS)
    if len(token_frame):
        display(token_frame.groupby(["model", "condition"]).agg(requests=("prompt_tokens", "size"),
                 mean_prompt_tokens=("prompt_tokens", "mean"), max_prompt_tokens=("prompt_tokens", "max"),
                 cached_answers=("cached", "sum")).reset_index())
        print("Answers still to generate:", int((~token_frame["cached"]).sum()))

## 6. Generate one model at a time

`eval()` disables training behaviour. Turning gradient checkpointing off avoids a training memory feature leaking into generation. `inference_mode()` prevents gradient construction, while greedy decoding chooses the highest-scoring next token each step. The user prompt is never shortened.

A result ending at the output limit without an end-of-sequence token is **truncated**, even if its first words look correct. Empty, failed and truncated attempts remain visible but are excluded from quality scores and invalidate that condition's headline. Rerunning retries those attempts; successful answers resume from cache.

In [ ]:
# PyTorch owns tensors/device memory; Transformers loads the causal LM; PEFT attaches one saved LoRA adapter.
# These imports happen only for an actual run. The notebook contains the actual loading code.
if RUN:
    import torch
    from transformers import AutoModelForCausalLM
    from peft import PeftModel
    assert torch.backends.mps.is_available(), "This configured experiment needs MPS"

def release_answer_memory():
    gc.collect()
    if RUN and torch.backends.mps.is_available():
        torch.mps.empty_cache()

def load_answer_model(model_id):
    spec = READY[model_id]["spec"]
    model = AutoModelForCausalLM.from_pretrained(spec["base_path"], dtype=torch.float32,
                                               local_files_only=True)
    if spec["kind"] == "local_adapter":
        model = PeftModel.from_pretrained(model, spec["adapter_path"], is_trainable=False,
                                         adapter_name="default", local_files_only=True)
        assert list(model.peft_config) == ["default"], "Only the root policy adapter belongs here"
    model.gradient_checkpointing_disable()
    model.eval()
    model.to(GEN["device"])
    assert model.training is False and not model.is_gradient_checkpointing
    return model

In [ ]:
# Generate exactly one answer and record what happened. The model returns prompt IDs followed by answer IDs.
def generate_answer(model, tokenizer, job):
    assert model.training is False and not model.is_gradient_checkpointing
    started = now()
    record = {"key": job["key"], "identity": job["identity"], "model": job["model"],
        "condition": job["condition"], "question_id": job["item"]["id"], "question": job["item"]["question"],
        "system_prompt": job["system_prompt"], "system_prompt_sha256": hashlib.sha256(job["system_prompt"].encode()).hexdigest(),
        "prompt_tokens": job["prompt_tokens"], "chunk_ids": [c["chunk_id"] for c in job["chunks"]],
        "chunks": [canonical_chunk(c) for c in job["chunks"]],
        "artifact": READY[job["model"]]["artifact"],
        "adapter_path": READY[job["model"]]["spec"].get("adapter_path"),
        "model_training": model.training, "gradient_checkpointing": bool(model.is_gradient_checkpointing),
        "started_at": started, "status": "error", "answer": "", "score": None}
    began = time.perf_counter()
    try:
        inputs = torch.tensor([job["input_ids"]], dtype=torch.long, device=GEN["device"])
        assert inputs.shape[1] <= GEN["max_input_tokens"]
        eos = model.generation_config.eos_token_id
        eos_ids = eos if isinstance(eos, list) else [eos]
        eos_ids = [value for value in eos_ids if value is not None]
        with torch.inference_mode():
            output = model.generate(input_ids=inputs, attention_mask=torch.ones_like(inputs),
                max_new_tokens=GEN["max_new_tokens"], do_sample=False, num_beams=1,
                eos_token_id=eos, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                use_cache=True)
        new_ids = output[0, inputs.shape[1]:].tolist()
        answer = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
        truncated = len(new_ids) >= GEN["max_new_tokens"] and (not new_ids or new_ids[-1] not in eos_ids)
        status = "truncated" if truncated else ("ok" if answer else "empty")
        record.update(answer=answer, status=status, completion_tokens=len(new_ids),
                      ended_with_eos=bool(new_ids and new_ids[-1] in eos_ids))
        if status == "ok":
            record["score"] = lab.score_answer(answer, job["item"], fidx)
    except Exception as exc:
        # Local errors have no provider credentials; keep the failure class and explanation reviewable.
        record.update(status="error", error_type=type(exc).__name__, error=str(exc))
    record.update(finished_at=now(), latency_seconds=time.perf_counter()-began)
    return record

In [ ]:
# Save after each question so interruption loses at most the active answer. Different identities are archived.
RESPONSES, RUN_FAILURES = {}, []
def persist_condition(model_id, condition):
    jobs = JOBS[(model_id, condition)]
    rows = [RESPONSES[job["key"]] for job in jobs if job["key"] in RESPONSES]
    expected = [job["item"]["id"] for job in jobs]
    valid = [r for r in rows if r["status"] == "ok"]
    complete = len(valid) == len(expected) and {r["question_id"] for r in valid} == set(expected)
    payload = {"identity": digest([job["key"] for job in jobs]), "provenance": PROVENANCE,
        "model": model_id, "condition": condition, "expected_question_ids": expected,
        "complete": complete, "status": "complete" if complete else "incomplete", "rows": rows, "answers": rows,
        "summary": lab.summarise([r["score"] for r in valid]) if complete and valid else None,
        "generation": GEN, "artifact": READY[model_id]["artifact"], "retrieval_identity": RETRIEVAL_HASH}
    save_result(OUTPUT_PATHS[(model_id, condition)], payload)

In [ ]:
# Run each checkpoint in isolation. Reuse successful rows and write progress after each answer.
if RUN:
    for model_id in SELECTED:
        if model_id not in READY:
            reason = next(r["path_or_reason"] for r in INVENTORY if r["model"] == model_id)
            for condition in CONDITIONS:
                save_result(OUTPUT_PATHS[(model_id, condition)], {"identity": digest([model_id, condition, reason, CACHE_PROVENANCE]),
                    "model": model_id, "condition": condition, "status": "SKIPPED", "reason": reason,
                    "expected_question_ids": [q["id"] for q in cohort(condition)], "complete": False, "rows": [], "answers": []})
            continue
        model_jobs = [job for condition in CONDITIONS for job in JOBS[(model_id, condition)]]
        for job in model_jobs:
            cached = read_cached(job)
            if cached is not None:
                RESPONSES[job["key"]] = cached
        pending = [job for job in model_jobs if job["key"] not in RESPONSES]
        model = None
        try:
            if pending:
                print("Loading", model_id, "fresh;", len(pending), "uncached requests")
                model = load_answer_model(model_id)
                for index, job in enumerate(pending, 1):
                    record = generate_answer(model, TOKENIZERS[model_id], job)
                    RESPONSES[job["key"]] = record
                    write_json(CACHE_DIR / (job["key"] + ".json"), record)
                    persist_condition(model_id, job["condition"])
                    print(f"{model_id} {index}/{len(pending)} {job['condition']} {job['item']['id']}: {record['status']}")
            else:
                print(model_id, "resumed entirely from successful caches; no weights loaded")
        except Exception as exc:
            RUN_FAILURES.append({"model": model_id, "error_type": type(exc).__name__, "error": str(exc)})
            print("Model failed:", model_id, type(exc).__name__, str(exc))
        finally:
            if model is not None:
                del model
            release_answer_memory()
        for condition in CONDITIONS:
            persist_condition(model_id, condition)
    display(pd.DataFrame(RUN_FAILURES))

## 7. Scoreboard: complete cohorts, uncertainty, and the declared bar

`lab.score_answer` and `lab.summarise` are unchanged. An answerable refusal is over-refusal; an unknown question needs a refusal. The rule-based scorer detects answer values and refusal phrases, not full semantic truth.

A **Wilson interval** estimates uncertainty around a proportion without producing impossible negative percentages. These are marginal intervals for individual metrics, not a paired significance test. Missing/failed answers make the entire model-condition headline **INCOMPLETE**. Oracle has no unknown questions: refusal-on-unknown metrics and their bar criterion are **N/A**.

In [ ]:
# Count the numerator and denominator explicitly; no questions means N/A, never a manufactured zero.
BAR = json.loads(project_path(MASTER["bar_path"]).read_text())["bar"]
METRICS = {"accuracy_answerable": (True, "correct"), "hallucination_rate_answerable": (True, "hallucinated"),
           "over_refusal_rate": (True, "refused"), "appropriate_refusal_rate": (False, "refused"),
           "hallucination_rate_unanswerable": (False, "hallucinated")}

def wilson(successes, n):
    if n == 0:
        return None, None
    z = NormalDist().inv_cdf((1 + MASTER["reporting"]["confidence"])/2)
    p = successes/n
    center = (p + z*z/(2*n))/(1+z*z/n)
    radius = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/(1+z*z/n)
    return max(0, center-radius), min(1, center+radius)

def result_rows(model_id, condition):
    return [RESPONSES[job["key"]] for job in JOBS.get((model_id, condition), []) if job["key"] in RESPONSES]

def complete_rows(model_id, condition):
    rows = result_rows(model_id, condition)
    expected = {q["id"] for q in cohort(condition)}
    valid = [r for r in rows if r["status"] == "ok"]
    if expected and len(valid) == len(expected) and {r["question_id"] for r in valid} == expected:
        return {r["question_id"]: r for r in valid}
    return None

In [ ]:
# Apply each threshold only when its denominator exists. N/A cannot become a full PASS.
def assess_bar(values):
    tests = {"accuracy": None if values["accuracy_answerable"] is None else values["accuracy_answerable"] >= BAR["accuracy_answerable"],
        "hallucination": None if values["hallucination_rate_answerable"] is None else values["hallucination_rate_answerable"] <= BAR["hallucination_rate_answerable"],
        "over_refusal": None if values["over_refusal_rate"] is None else values["over_refusal_rate"] <= BAR["max_over_refusal"],
        "appropriate_refusal": None if values["appropriate_refusal_rate"] is None else values["appropriate_refusal_rate"] >= BAR["appropriate_refusal_rate"]}
    overall = "FAIL" if False in tests.values() else ("N/A (partial bar)" if None in tests.values() else "PASS")
    return overall, {name: "N/A" if value is None else "PASS" if value else "FAIL" for name, value in tests.items()}

In [ ]:
# Only complete cohorts receive headline metrics; missing denominators keep their N/A label.
summary_rows, INTERVALS = [], []
if RUN:
    for model_id in SELECTED:
        for condition in CONDITIONS:
            complete = complete_rows(model_id, condition)
            rows = result_rows(model_id, condition)
            state = "SKIPPED" if model_id not in READY else "N/A (no eligible questions)" if not cohort(condition) else "COMPLETE" if complete else "INCOMPLETE"
            row = {"model": model_id, "label": SPECS[model_id]["label"], "condition": condition, "status": state,
                   "expected": len(cohort(condition)), "successful": sum(r["status"] == "ok" for r in rows),
                   "mean_prompt_tokens": sum(r["prompt_tokens"] for r in rows)/len(rows) if rows else None,
                   "bar": "N/A", "bar_tests": {}}
            if complete:
                row.update(lab.summarise([r["score"] for r in complete.values()]))
            for metric, (answerable, field) in METRICS.items():
                group = [r["score"] for r in complete.values() if r["score"]["answerable"] == answerable] if complete else []
                n, k = len(group), sum(r[field] for r in group)
                low, high = wilson(k, n)
                row[metric] = row.get(metric) if n else None
                row[metric+"_low"], row[metric+"_high"] = low, high
                row[metric+"_ci"] = f"{100*low:.1f}–{100*high:.1f}%" if n else "N/A"
                INTERVALS.append({"model": model_id, "condition": condition, "metric": metric, "successes": k,
                                  "denominator": n, "low": low, "high": high})
            if complete:
                row["bar"], row["bar_tests"] = assess_bar(row)
            summary_rows.append(row)
metrics_frame = pd.DataFrame(summary_rows)
if len(metrics_frame):
    display(metrics_frame[["model", "condition", "status", "expected", "successful", "bar", "mean_prompt_tokens",
                           *[column for metric in METRICS for column in [metric, metric+"_ci"]]]])
    print("Bar thresholds:", BAR)
    display(pd.DataFrame([{**{k: r[k] for k in ["model", "condition"]}, **r["bar_tests"]} for r in summary_rows]))
else:
    print("Preview: no new quality scores. Set run: true after reviewing the plan.")

In [ ]:
# Matplotlib displays complete-cohort rates as heatmaps. Grey N/A cells are not zeros.
FIGURE_PATHS = []
if len(metrics_frame):
    import matplotlib.pyplot as plt
    import numpy as np
    for metric, title in [("accuracy_answerable", "Answerable accuracy"), ("appropriate_refusal_rate", "Appropriate refusal on unknowns")]:
        table = metrics_frame.pivot(index="model", columns="condition", values=metric).reindex(index=SELECTED, columns=CONDITIONS)
        values = table.to_numpy(dtype=float)
        fig, ax = plt.subplots(figsize=(max(8, len(CONDITIONS)*1.7), max(3, len(SELECTED)*.7)))
        palette = plt.get_cmap("YlGn").copy()
        palette.set_bad("#e5e7eb")
        plotted = ax.imshow(np.ma.masked_invalid(values), vmin=0, vmax=1, cmap=palette, aspect="auto")
        ax.set_xticks(range(len(CONDITIONS)), CONDITIONS, rotation=25, ha="right")
        ax.set_yticks(range(len(SELECTED)), [SPECS[m]["label"] for m in SELECTED])
        for i in range(len(SELECTED)):
            for j in range(len(CONDITIONS)):
                value = values[i, j]
                ax.text(j, i, "N/A" if np.isnan(value) else f"{value:.0%}", ha="center", va="center", color="black")
        ax.set_title(title + " · complete cohorts only")
        fig.colorbar(plotted, ax=ax, label="Rate")
        fig.tight_layout()
        if MASTER["reporting"]["save_figures"]:
            path = RESULT_DIR / (metric + ".png")
            fig.savefig(path, dpi=150)
            FIGURE_PATHS.append(str(path))
        plt.show()
        plt.close(fig)

## 8. What each technique adds: compare the same questions

A positive delta is a percentage-point improvement in correct behaviour. For known questions, that means answering correctly; for unknown questions, it means refusing appropriately. **Fixed** means wrong before/right after. **Broke** means right before/wrong after.

Pairs use the same predeclared question IDs on both sides and require complete cohorts. We never improve a delta by dropping failed requests. Oracle comparisons use its answerable cohort only. The aggregate correct-behaviour delta mixes two behaviours; the separate answerable and unknown deltas keep the distinction visible.

In [ ]:
# Compare completed pairs; no inference is made from an incomplete or differently sized cohort.
PAIRED, PAIR_DETAILS = [], []
def compare_pair(left, right, comparison):
    before, after = complete_rows(*left), complete_rows(*right)
    row = {"comparison": comparison, "before": "/".join(left), "after": "/".join(right)}
    if not before or not after or set(before) != set(after):
        PAIRED.append({**row, "status": "N/A: incomplete or different cohorts", "n": None})
        return
    ids = sorted(before)
    fixed = [qid for qid in ids if not before[qid]["score"]["correct"] and after[qid]["score"]["correct"]]
    broke = [qid for qid in ids if before[qid]["score"]["correct"] and not after[qid]["score"]["correct"]]
    row.update(status="matched complete", n=len(ids), fixed=len(fixed), broke=len(broke), delta_pp=100*(len(fixed)-len(broke))/len(ids))
    for label, answerable in [("answerable", True), ("unknown", False)]:
        subset = [qid for qid in ids if by_id[qid]["answerable"] == answerable]
        row[label+"_n"] = len(subset)
        row[label+"_delta_pp"] = 100*sum(int(after[qid]["score"]["correct"])-int(before[qid]["score"]["correct"]) for qid in subset)/len(subset) if subset else None
    PAIRED.append(row)
    for outcome, qids in [("fixed", fixed), ("broke", broke)]:
        for qid in qids:
            PAIR_DETAILS.append({"comparison": comparison, "before": row["before"], "after": row["after"], "outcome": outcome,
                "question_id": qid, "question": by_id[qid]["question"], "answerable": by_id[qid]["answerable"],
                "before_answer": before[qid]["answer"], "after_answer": after[qid]["answer"]})

if RUN:
    for condition in CONDITIONS:
        for first, second in [("base", "sft"), ("sft", "sft_dpo"), ("cpt", "cpt_sft_merged")]:
            if first in SELECTED and second in SELECTED:
                compare_pair((first, condition), (second, condition), f"{first} → {second}")
    for model_id in SELECTED:
        for first, second in [("engineered", "with_docs"), ("with_docs", "rag_rerank"), ("rag_similarity", "rag_rerank")]:
            if first in CONDITIONS and second in CONDITIONS:
                compare_pair((model_id, first), (model_id, second), f"{first} → {second}")
    display(pd.DataFrame(PAIRED))
    if MASTER["reporting"]["show_paired_details"]:
        with pd.option_context("display.max_colwidth", None):
            display(pd.DataFrame(PAIR_DETAILS))

In [ ]:
# Exact-string equality detects identical outputs; different strings can still mean the same thing.
IDENTICAL = []
if RUN:
    for condition in CONDITIONS:
        for first, second in combinations(SELECTED, 2):
            a, b = complete_rows(first, condition), complete_rows(second, condition)
            matched = bool(a and b and set(a) == set(b))
            count = sum(a[qid]["answer"] == b[qid]["answer"] for qid in a) if matched else None
            IDENTICAL.append({"condition": condition, "model_a": first, "model_b": second,
                              "n": len(a) if matched else None, "identical_answers": count,
                              "status": "matched complete" if matched else "N/A: incomplete"})
    display(pd.DataFrame(IDENTICAL))

## 9. Refusal generalization and RAG failure diagnosis

Unknown questions have no gold fact, so they never get a fabricated retrieval-miss label. Inspect their refusal categories separately. On a wrong **answerable** RAG response, missing gold context is a retrieval miss; present gold context means a generation error under this rubric. This diagnosis is useful but not a proof of causal responsibility.

Oracle supplies gold chunks directly. Its answerable accuracy estimates what this model can do with the needed evidence present; it says nothing about refusing unknown questions.

In [ ]:
# A category rate is a subset of a complete condition, with its own denominator and Wilson interval.
REFUSAL_CATEGORIES = []
if RUN:
    unknown_categories = sorted({q["category"] for q in items if not q["answerable"]})
    for model_id in SELECTED:
        for condition in CONDITIONS:
            complete = complete_rows(model_id, condition)
            for category in unknown_categories:
                qids = [q["id"] for q in cohort(condition) if not q["answerable"] and q["category"] == category]
                valid = bool(complete and qids)
                n = len(qids) if valid else 0
                k = sum(complete[qid]["score"]["refused"] for qid in qids) if valid else 0
                low, high = wilson(k, n)
                REFUSAL_CATEGORIES.append({"model": model_id, "condition": condition, "category": category,
                    "expected": len(qids), "n": n, "refused": k if valid else None,
                    "appropriate_refusal": k/n if valid else None, "low": low, "high": high,
                    "status": "complete" if valid else "N/A"})
    display(pd.DataFrame(REFUSAL_CATEGORIES))

In [ ]:
# Read the exact chunks used for each answer, rather than searching again during error attribution.
ATTRIBUTION, ORACLE_CEILING = [], []
if RUN:
    for model_id in SELECTED:
        for condition in [c for c in CONDITIONS if c.startswith("rag_")]:
            for row in result_rows(model_id, condition):
                q = by_id[row["question_id"]]
                if row["status"] == "ok" and q["answerable"] and not row["score"]["correct"]:
                    present = any(q["fact_id"] in c["fact_ids"] for c in row["chunks"])
                    ATTRIBUTION.append({"model": model_id, "condition": condition, "question_id": q["id"],
                        "question": q["question"], "answer": row["answer"], "chunk_ids": row["chunk_ids"],
                        "error": "generation error" if present else "retrieval miss"})
        if "oracle" in CONDITIONS:
            oracle = complete_rows(model_id, "oracle")
            for condition in [c for c in CONDITIONS if c.startswith("rag_")]:
                rag_rows = complete_rows(model_id, condition)
                ids = {q["id"] for q in cohort("oracle")}
                matched = bool(oracle and rag_rows and set(oracle) == ids and ids <= set(rag_rows))
                ORACLE_CEILING.append({"model": model_id, "condition": condition, "status": "matched answerable subset" if matched else "N/A: incomplete",
                    "n_answerable": len(ids) if matched else None,
                    "rag_accuracy": sum(rag_rows[qid]["score"]["correct"] for qid in ids)/len(ids) if matched else None,
                    "oracle_accuracy": sum(oracle[qid]["score"]["correct"] for qid in ids)/len(ids) if matched else None,
                    "delta_pp": 100*sum(int(oracle[qid]["score"]["correct"])-int(rag_rows[qid]["score"]["correct"]) for qid in ids)/len(ids) if matched else None})
    attribution_frame = pd.DataFrame(ATTRIBUTION)
    if len(attribution_frame):
        display(attribution_frame.groupby(["model", "condition", "error"]).size().rename("wrong_answers").reset_index())
        with pd.option_context("display.max_colwidth", None):
            display(attribution_frame)
    else:
        print("No scored wrong answerable RAG responses in this run.")
    display(pd.DataFrame(ORACLE_CEILING))

## 10. Inspect one question across every model and condition

Set `reporting.inspector_question_id` in YAML, or call `inspect_question("your-id")` after the run. The display includes full answers, scoring flags, prompt sizes and chunk IDs. An unknown question correctly shows no oracle answer. A question outside a smoke cohort shows **not requested**, not a failed model.

In [ ]:
# Build a wide side-by-side answer table, with a separate diagnostics table underneath.
def inspect_question(question_id):
    if question_id not in by_id:
        raise KeyError(f"Unknown question ID: {question_id}")
    q = by_id[question_id]
    display(Markdown(f"**{question_id}** · {q['question']}"))
    display(pd.DataFrame([{key: q.get(key) for key in ["answerable", "category", "style", "fact_id", "expected_value", "unit", "near_miss_of", "absence_check"]}]))
    answers, details = [], []
    for model_id in SELECTED:
        for condition in CONDITIONS:
            found = next((r for r in result_rows(model_id, condition) if r["question_id"] == question_id), None)
            status = found["status"] if found else "N/A: oracle has no unknowns" if condition == "oracle" and not q["answerable"] else "not requested / unavailable"
            answers.append({"model": model_id, "condition": condition, "answer": found["answer"] if found else status})
            details.append({"model": model_id, "condition": condition, "status": status,
                            "prompt_tokens": found.get("prompt_tokens") if found else None,
                            "chunk_ids": found.get("chunk_ids") if found else None,
                            "score": found.get("score") if found else None})
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(answers).pivot(index="model", columns="condition", values="answer").reindex(index=SELECTED, columns=CONDITIONS))
        display(pd.DataFrame(details))
if RUN:
    inspect_question(MASTER["reporting"]["inspector_question_id"] or items[0]["id"])

## How to read this

- With the full set, one answerable question moves accuracy by **3.3 percentage points**; one unknown question moves appropriate refusal by **5 points**. A smoke subset is much noisier.
- Overlapping Wilson intervals mean these marginal intervals have **not shown a difference**. Overlap alone is not a paired statistical test; non-overlap is not a replacement for inspecting fixed/broke examples either.
- The bar uses point estimates, not confidence bounds. **PASS** means passing the four predeclared thresholds on this development cohort, not a guarantee about customers. Oracle cannot pass the full bar because it has no unknown questions.
- The scorer checks values and refusal phrases; its numeric matching ignores the unit argument, so a unit error can pass. It can miss paraphrases, accept an otherwise wrong answer containing the target value, or misread mixed answer/refusal text. Colour lists and Arc's zero-valued fast-charge fact are excluded from `dev_v1`, but the scoring implementation is unchanged.
- A missing/truncated/failed request must be rerun before that cohort gets a headline. Do not compare an incomplete run with a complete one.
- `dev_v1` was informed by earlier development failures. It is **still a development set**, not independent final evidence. `evals/heldout.json` remains reserved for notebook 08.

## Appendix · historical, not comparable

These older files use different question sets, prompts, or execution settings. They are shown only as historical context; no percentage-point deltas connect them to this scoreboard. In particular, `runs/08_sft_eval.json` behaves like a **with-documents** run despite its name. Files named as held-out results stay outside this development notebook.

In [ ]:
# Read only stored summary fields from older top-level runs; never mix their rows into the master cohort.
HISTORICAL = []
for path in sorted(ROOT.glob(MASTER["reporting"]["historical_glob"])):
    if any(fragment.lower() in path.name.lower() for fragment in MASTER["reporting"]["historical_exclude_name_fragments"]):
        continue
    try:
        saved = json.loads(path.read_text())
    except (OSError, ValueError):
        continue
    summary = saved.get("summary") if isinstance(saved, dict) else None
    if isinstance(summary, dict):
        HISTORICAL.append({"file": str(path.relative_to(ROOT)), "comparability": "HISTORICAL — not comparable",
            "note": "Actually uses documents" if path.name == "08_sft_eval.json" else "Different cohort/prompt/settings",
            **{key: value for key, value in summary.items() if isinstance(value, (str, int, float, bool)) or value is None}})
display(pd.DataFrame(HISTORICAL))

## Save the comparison

The JSON retains identities, denominators, intervals, diagnostics and historical labels. The CSV is the current scoreboard for sorting elsewhere. Changed identities under the same tag archive the previous summary; a different tag requires a new output directory. The notebook file itself ships with empty outputs.

In [ ]:
# Write only into this run's configured namespace. Keep database connection strings out of saved settings.
if RUN:
    safe_settings = copy.deepcopy(MASTER)
    safe_settings["postgres"]["local_dsn"] = "[configured in master_eval.yaml]"
    summary_payload = {"identity": digest({"jobs": {"/".join(key): [j["key"] for j in value] for key, value in JOBS.items()},
                                           "inventory": INVENTORY, "bar": BAR, "reporting": MASTER["reporting"]}),
        "provenance": PROVENANCE, "settings": safe_settings, "inventory": INVENTORY, "plan": PLAN,
        "selected_question_ids": [q["id"] for q in items], "scoreboard": summary_rows, "intervals": INTERVALS,
        "retrieval": {"identity": RETRIEVAL_HASH, "index_hash": INDEX_HASH, "metrics": retrieval_rows,
                      "misses": retrieval_misses, "local_usage": RETRIEVAL_STATS, "snapshot": str(RETRIEVAL_PATH) if RETRIEVAL_PATH else None},
        "paired": PAIRED, "fixed_broke": PAIR_DETAILS, "identical_answers": IDENTICAL,
        "refusal_categories": REFUSAL_CATEGORIES, "rag_errors": ATTRIBUTION, "oracle_ceiling": ORACLE_CEILING,
        "historical_not_comparable": HISTORICAL, "run_failures": RUN_FAILURES, "figures": FIGURE_PATHS,
        "completion": "complete" if summary_rows and all(r["status"] in {"COMPLETE", "SKIPPED", "N/A (no eligible questions)"} for r in summary_rows) else "incomplete"}
    save_result(RESULT_DIR / "summary.json", summary_payload)
    metrics_frame.to_csv(RESULT_DIR / "summary.csv", index=False)
    print("Saved", RESULT_DIR / "summary.json")
    print("Saved", RESULT_DIR / "summary.csv")